# Plotting Gaps — Scenario Gallery

Each section below renders **one inline-matplotlib pattern** that currently lives in notebooks instead of in `scripts/plotting/`. The point is to see every scenario side-by-side so we can decide what to promote into the module.

Gaps covered (see `AGENTS.md` → "Known gaps"):

1. Seasonal / region shading (Ethiopia seasons)
2. Threshold / reference lines (flow ratio, smooth/raw)
3. Event date markers (flow-fix boundaries)
4. Uncertainty / percentile bands
5. PMF date-range highlight
6. AAE source-region shading (fossil / mixed / biomass)
7. Excluded-sample overlay on scatter
8. Site-color centralization (anti-pattern demo)

Data here is **synthetic** so the notebook runs anywhere. Each scenario notes how to swap in real data via `scripts/` imports.

In [ ]:
# Standard setup — the same pattern AGENTS.md recommends for real notebooks.
# This notebook lives at repo ROOT, so we point sys.path at the scripts
# package under research/ftir_hips_chem/. Notebooks that live INSIDE
# research/ftir_hips_chem/ should use './scripts' instead.
#
# NOTE: we do NOT call plt.style.use('seaborn-v0_8-darkgrid'). Matching the
# convention in Analysis_Tasks_Jan2025.ipynb, we rely on matplotlib's default
# (white background) so figures are clean and print/publish well.
import sys
from pathlib import Path

sys.path.insert(0, './research/ftir_hips_chem/scripts')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates

# Try to pull from the real module; fall back to synthetic-only if not available
try:
    from config import SITES, FLOW_FIX_PERIODS
    HAVE_CONFIG = True
except Exception as e:
    print(f'config not importable ({e}); scenarios will use hardcoded stand-ins')
    HAVE_CONFIG = False
    SITES = {
        'Beijing':     {'color': '#E74C3C'},
        'Delhi':       {'color': '#3498DB'},
        'JPL':         {'color': '#2ECC71'},
        'Addis_Ababa': {'color': '#F39C12'},
    }
    FLOW_FIX_PERIODS = {
        'JPL': {'before_end': '2022-09-30', 'after_start': '2023-05-01',
                'has_before_data': True}
    }

# Keep only non-style rcParams — size & font are fine to standardize
plt.rcParams['figure.figsize'] = (11, 4.5)
plt.rcParams['font.size'] = 10

np.random.seed(7)

In [ ]:
# Shared synthetic data — 15 months of Addis daily BC and JPL daily flow ratio
dates_addis = pd.date_range('2023-09-01', '2024-12-01', freq='D')
month = dates_addis.month

# Seasonal mean: dry higher, Kiremt lower (rainy washout)
seasonal_mean = np.where(np.isin(month, [6, 7]), 9.0,
                 np.where(np.isin(month, [8, 11, 12]), 5.5, 12.0))
bc_addis = seasonal_mean + np.random.normal(0, 2.5, len(dates_addis))
bc_addis = np.clip(bc_addis, 0.5, None)
df_addis = pd.DataFrame({'bc_ir': bc_addis}, index=dates_addis)

# JPL flow ratio: drops from ~2.2 to ~1.0 across a flow-fix gap
dates_jpl = pd.date_range('2022-01-01', '2024-06-01', freq='D')
flow = np.where(dates_jpl < pd.Timestamp('2022-10-01'), 2.15,
         np.where(dates_jpl < pd.Timestamp('2023-05-01'), np.nan, 1.04))
flow = flow + np.random.normal(0, 0.08, len(dates_jpl))
df_jpl = pd.DataFrame({'flow_ratio': flow}, index=dates_jpl)

# Matched filter/aeth scatter with a few outliers
n = 220
ec = np.random.gamma(2.0, 1.3, n)
aeth_bc = 0.92 * ec + np.random.normal(0, 0.6, n)
is_excluded = np.zeros(n, dtype=bool)
# Plant a couple of contamination-like outliers
idx = np.random.choice(n, size=6, replace=False)
aeth_bc[idx] += np.random.uniform(6, 11, 6)
is_excluded[idx] = True
df_scatter = pd.DataFrame({'ec': ec, 'aeth_bc': aeth_bc, 'is_excluded': is_excluded})

print(f'df_addis: {df_addis.shape}, df_jpl: {df_jpl.shape}, df_scatter: {df_scatter.shape}')

---
## Scenario 1 — Seasonal shading (Ethiopia)

**What notebooks do today:** each notebook redefines `SEASON_COLORS` and loops `ax.axvspan(start, end, color=...)` per season window.

**Proposed fix:** `plotting.overlays.add_seasonal_shading(ax, seasons=ETHIOPIA_SEASONS)` + move the color dict into `config.ETHIOPIA_SEASONS`.

Found in: `addis_02_temporal_patterns`, `addis_02.5_temporal_patterns_filter`, `addis_02_temporal_patterns_daily`.

In [ ]:
# INLINE PATTERN being repeated across notebooks ----
SEASON_COLORS = {
    'Dry Season':         '#E67E22',
    'Belg Rainy Season':  '#27AE60',
    'Kiremt Rainy Season':'#3498DB',
}
SEASON_MONTHS = {
    'Dry Season':         [1, 2, 3, 4, 5, 9, 10],
    'Belg Rainy Season':  [6, 7],
    'Kiremt Rainy Season':[8, 11, 12],
}

def _season_of(m):
    for name, months in SEASON_MONTHS.items():
        if m in months:
            return name

fig, ax = plt.subplots()
ax.plot(df_addis.index, df_addis['bc_ir'], color='#222', lw=0.8)

# Each notebook currently reimplements this loop
season_series = pd.Series(df_addis.index.month, index=df_addis.index).map(_season_of)
runs = (season_series != season_series.shift()).cumsum()
for _, grp in season_series.groupby(runs):
    ax.axvspan(grp.index[0], grp.index[-1], color=SEASON_COLORS[grp.iloc[0]], alpha=0.18, zorder=0)

# Hand-rolled legend (also inline in every notebook)
from matplotlib.patches import Patch
handles = [Patch(facecolor=c, alpha=0.35, label=n) for n, c in SEASON_COLORS.items()]
ax.legend(handles=handles, loc='upper right', fontsize=8)

ax.set(title='Scenario 1 — Addis BC with seasonal shading (inline)',
       xlabel='Date', ylabel='BC IR (µg/m³)')
plt.tight_layout(); plt.show()

---
## Scenario 2 — Threshold / reference lines

**What notebooks do today:** `ax.axhline(1.0, ...)` and `ax.axhline(2.0, ...)` hardcoded; same for smooth/raw `[1, 2.5, 4, 5]%` vlines.

**Proposed fix:** `plotting.overlays.add_threshold_line(ax, value, axis='h', label=...)` with smart default label formatting.

Currently **partially** covered inside `timeseries.flow_ratio()` (hardcoded) and `distributions.smooth_raw_histogram()` (hardcoded list).

In [ ]:
fig, ax = plt.subplots()
ax.plot(df_jpl.index, df_jpl['flow_ratio'], color=SITES['JPL']['color'], lw=0.8)

# Hardcoded thresholds — currently live inside plotting.timeseries.flow_ratio too
ax.axhline(1.0, color='green', ls='--', lw=1, alpha=0.8, label='Ideal (1.0)')
ax.axhline(2.0, color='red',   ls='--', lw=1, alpha=0.8, label='Degraded (2.0)')

ax.set(title='Scenario 2 — JPL flow ratio with threshold lines (inline)',
       xlabel='Date', ylabel='Flow ratio', ylim=(0.5, 2.8))
ax.legend(loc='upper right')
plt.tight_layout(); plt.show()

---
## Scenario 3 — Event date markers (flow-fix boundaries)

**What notebooks do today:** `ax.axvline(pd.to_datetime(FLOW_FIX_PERIODS[site]['before_end']), ...)` — good that we read from config, but the vline + annotation is re-typed every time.

**Proposed fix:** `plotting.overlays.add_event_marker(ax, date, label='Flow fix', color='red')` and/or `plotting.overlays.add_flow_fix_boundary(ax, site)` that reads `FLOW_FIX_PERIODS` directly.

Found in: `FlowFix_BeforeAfter_Analysis`, `Multi_Site_Analysis_FollowUp`.

In [ ]:
fig, ax = plt.subplots()
ax.plot(df_jpl.index, df_jpl['flow_ratio'], color=SITES['JPL']['color'], lw=0.8)

# Inline pattern: pull two dates from config, draw two vlines, shade the gap
ff = FLOW_FIX_PERIODS['JPL']
bef = pd.to_datetime(ff['before_end'])
aft = pd.to_datetime(ff['after_start'])
ax.axvline(bef, color='red',   ls='--', lw=1.2, label=f"before_end ({bef:%Y-%m-%d})")
ax.axvline(aft, color='green', ls='--', lw=1.2, label=f"after_start ({aft:%Y-%m-%d})")
ax.axvspan(bef, aft, color='gray', alpha=0.15, label='Gap (no data)')

ax.set(title='Scenario 3 — JPL flow-fix event markers (inline)',
       xlabel='Date', ylabel='Flow ratio')
ax.legend(loc='upper right', fontsize=8)
plt.tight_layout(); plt.show()

---
## Scenario 4 — Uncertainty / percentile bands

**What notebooks do today:** `ax.fill_between(x, q25, q75, alpha=0.08)` for p25–p75 ribbon + median line.

**Proposed fix:** `plotting.overlays.add_uncertainty_band(ax, x, lower, upper, alpha=0.2)` — optionally takes a DataFrame and column names so callers don't pre-compute quantiles.

Found in: `smoothing_comparison`, `smoothing_comparison_executed`, `addis_05_diurnal_wavelength_analysis`.

In [ ]:
# Diurnal rollup — percentile band around the median
hours = np.arange(24)
# Fake a diurnal cycle + daily replicates for quantiles
daily = np.array([5 + 4*np.sin((h - 6) * np.pi / 12) + np.random.normal(0, 1.5, 90) for h in hours]).T
q25 = np.percentile(daily, 25, axis=0)
q50 = np.percentile(daily, 50, axis=0)
q75 = np.percentile(daily, 75, axis=0)

fig, ax = plt.subplots()
ax.fill_between(hours, q25, q75, color='#8E44AD', alpha=0.18, label='p25–p75')
ax.plot(hours, q50, color='#8E44AD', lw=2, label='median')
ax.set(title='Scenario 4 — Diurnal uncertainty band (inline)',
       xlabel='Hour of day', ylabel='BC IR (µg/m³)', xticks=range(0, 24, 3))
ax.legend()
plt.tight_layout(); plt.show()

---
## Scenario 5 — PMF / analysis-window date-range highlight

**What notebooks do today:** `ax.axvspan(pmf_min, pmf_max, alpha=0.08)` to mark the date range covered by PMF output.

**Proposed fix:** same `add_period_shading(ax, start, end, label='PMF period')` helper used by seasons, parameterized.

Found in: `ETAD_Factor_Analysis`.

In [ ]:
pmf_min = pd.Timestamp('2024-02-01')
pmf_max = pd.Timestamp('2024-08-01')

fig, ax = plt.subplots()
ax.plot(df_addis.index, df_addis['bc_ir'], color='#222', lw=0.8)
ax.axvspan(pmf_min, pmf_max, color='#9B59B6', alpha=0.12, label=f'PMF window ({pmf_min:%b %Y} – {pmf_max:%b %Y})')
ax.set(title='Scenario 5 — PMF analysis window highlight (inline)',
       xlabel='Date', ylabel='BC IR (µg/m³)')
ax.legend()
plt.tight_layout(); plt.show()

---
## Scenario 6 — AAE source-region shading

**What notebooks do today:** three `axvspan` calls on an AAE histogram to mark fossil-fuel / mixed / biomass regions.

**Proposed fix:** `plotting.overlays.add_aae_regions(ax, fossil_max=0.9, biomass_min=1.4)` that also labels each band.

Found in: `addis_01_source_apportionment`, `addis_01_source_apportionment_daily`.

In [ ]:
aae = np.concatenate([
    np.random.normal(0.85, 0.08, 250),   # fossil-fuel-dominated
    np.random.normal(1.15, 0.10, 400),   # mixed
    np.random.normal(1.55, 0.12, 180),   # biomass-dominated
])

fig, ax = plt.subplots()
ax.hist(aae, bins=50, color='#34495E', edgecolor='white')

# Inline region bands
ax.axvspan(0.5, 0.9, color='#E74C3C', alpha=0.15, label='Fossil fuel (<0.9)')
ax.axvspan(0.9, 1.4, color='#F1C40F', alpha=0.15, label='Mixed (0.9–1.4)')
ax.axvspan(1.4, 2.0, color='#27AE60', alpha=0.15, label='Biomass (>1.4)')

ax.set(title='Scenario 6 — AAE source-region shading (inline)',
       xlabel='AAE', ylabel='Count', xlim=(0.5, 2.0))
ax.legend()
plt.tight_layout(); plt.show()

---
## Scenario 7 — Excluded-sample overlay on a scatter

**What the module does today:** `crossplots.scatter(..., outlier_col='is_excluded')` highlights outliers as red X on the *before* panel of `before_after_outliers`, but for a single-panel scatter the default drops them entirely.

**Proposed fix:** `crossplots.scatter(..., show_excluded=True, excluded_marker='X', excluded_alpha=0.6)` — draw clean + excluded on the same axes so reviewers can see what was removed without flipping between panels.

Currently 0 notebooks do this; they either hide excluded or render side-by-side. Showing an overlay makes exclusion review one glance instead of three.

In [ ]:
clean = df_scatter[~df_scatter['is_excluded']]
excl  = df_scatter[ df_scatter['is_excluded']]

fig, ax = plt.subplots(figsize=(6, 6))
ax.scatter(clean['ec'], clean['aeth_bc'], s=36, color='#2C3E50', alpha=0.65, label=f'Clean (n={len(clean)})')
ax.scatter(excl['ec'],  excl['aeth_bc'],  s=110, color='#E74C3C', marker='X', edgecolor='black', lw=0.8, label=f'Excluded (n={len(excl)})')

# 1:1 reference
lim = max(df_scatter['ec'].max(), df_scatter['aeth_bc'].max()) * 1.05
ax.plot([0, lim], [0, lim], color='gray', ls=':', lw=1, label='1:1')

ax.set(title='Scenario 7 — Clean + excluded overlay (proposed)',
       xlabel='Filter EC (µg/m³)', ylabel='Aeth BC (µg/m³)',
       xlim=(0, lim), ylim=(0, lim))
ax.set_aspect('equal')
ax.legend(loc='upper left')
plt.tight_layout(); plt.show()

---
## Scenario 8 — Site colors, centralized vs. ad-hoc (anti-pattern demo)

**What notebooks sometimes do:** redefine `site_colors = {'Beijing': 'red', ...}` locally, which drifts from `config.SITES[site]['color']`.

**Proposed fix:** nothing new — this already works. The demo is just a reminder: read from `SITES[site]['color']` or `PlotConfig.get_site_color(site)`.

Left panel = correct (pulls from config). Right panel = wrong (local dict drifts; Delhi shown in wrong color).

In [ ]:
sites = ['Beijing', 'Delhi', 'JPL', 'Addis_Ababa']
vals  = [42, 38, 15, 28]

fig, axes = plt.subplots(1, 2, figsize=(11, 4))

# LEFT: canonical (read from SITES)
axes[0].bar(sites, vals, color=[SITES[s]['color'] for s in sites])
axes[0].set_title('Canonical — reads SITES[s]["color"]')
axes[0].tick_params(axis='x', labelrotation=20)

# RIGHT: anti-pattern — local redefinition that drifts
local_colors = {'Beijing': '#E74C3C', 'Delhi': '#9B59B6',  # <-- wrong purple, should be blue
                'JPL': '#2ECC71', 'Addis_Ababa': '#F39C12'}
axes[1].bar(sites, vals, color=[local_colors[s] for s in sites])
axes[1].set_title('Anti-pattern — local dict drifts')
axes[1].tick_params(axis='x', labelrotation=20)

plt.tight_layout(); plt.show()

---
## Summary — what to build

Based on the scenarios above, the minimum viable `plotting/overlays.py` is:

```python
# plotting/overlays.py
def add_seasonal_shading(ax, seasons=ETHIOPIA_SEASONS, alpha=0.18, legend=True): ...
def add_period_shading(ax, start, end, color, alpha=0.12, label=None): ...
def add_threshold_line(ax, value, axis='h', color='red', ls='--', label=None): ...
def add_event_marker(ax, date, color='red', ls='--', label=None): ...
def add_flow_fix_boundary(ax, site): ...                 # reads config.FLOW_FIX_PERIODS
def add_uncertainty_band(ax, x, lower, upper, alpha=0.2, color=None, label=None): ...
def add_aae_regions(ax, fossil_max=0.9, biomass_min=1.4, alpha=0.15): ...
```

Plus one `config.py` addition:

```python
ETHIOPIA_SEASONS = {
    'Dry Season':          {'color': '#E67E22', 'months': [1,2,3,4,5,9,10]},
    'Belg Rainy Season':   {'color': '#27AE60', 'months': [6,7]},
    'Kiremt Rainy Season': {'color': '#3498DB', 'months': [8,11,12]},
}
```

And one `crossplots.scatter` extension: `show_excluded=True` kwarg.

With those in place, the inline gymnastics in `addis_01`, `addis_02*`, `FlowFix_BeforeAfter`, `ETAD_Factor_Analysis`, `smoothing_comparison`, and `addis_05_*` all collapse to one or two lines per plot.